In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

import os
import glob

# Use the Java/Hadoop installation configured in your environment.
# Use the Java/Hadoop installation configured in your environment.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('TLC_FHVHV_Processing') \
    .master('local[*]') \
    .config("spark.driver.memory", "8g") \
    .config('spark.sql.parquet.enableVectorizedReader', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version: {spark.version}')
print('Environment ready!')

    # .config("spark.driver.maxResultSize", "3g") \
    # .config("spark.sql.shuffle.partitions", "32") \
    # .config("spark.memory.fraction", "0.8") \
    # .config("spark.memory.storageFraction", "0.2") \

# 02b. Hourly preparation: 2024–2025

This stage prepares the 24 monthly inputs required by the analysis. It retains the original cleaning and metric formulas, while removing exploratory and unused aggregate branches. Existing selected outputs are overwritten when this notebook is executed; use a separate output directory for a fresh reproduction.


## Select 2024–2025 inputs


In [ ]:
RAW_DATA_PATH = str(RAW_DIR)
OUTPUT_PATH = str(PROCESSED_DIR)

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Policy cutoff date
POLICY_DATE = '2025-01-05'

# Load 2024-2025 parquet files
TARGET_YEARS = ['2024', '2025']
all_files = glob.glob(os.path.join(RAW_DATA_PATH, 'fhvhv_*.parquet'))
target_files = sorted([
    f for f in all_files
    if any(year in os.path.basename(f) for year in TARGET_YEARS)
])

print(f'Files to load: {len(target_files)}')
print(f'Output path:   {OUTPUT_PATH}')

## Load monthly source records


In [ ]:
print('Loading raw FHVHV parquet files...')
# df = spark.read.parquet(*target_files)


df = None
for file in target_files:
    temp_df = spark.read.parquet(file)

    if df is None:
        df = temp_df
    else:
        df = df.unionByName(temp_df,allowMissingColumns=True)
    print(f"loaded:{file} | Rows: {temp_df.count()}")

raw_count = df.count()
print(f'Raw rows loaded: {raw_count:,}')

## Calculate cleaning diagnostics


In [ ]:
df = df\
    .withColumn('trip_time_cal',F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) \
    .withColumn('trip_speed',(F.col('trip_miles')/F.col('trip_time'))*3600)

## Clean trips


In [ ]:
df_clean = df \
    .dropDuplicates() \
    .filter(F.col('trip_miles') > 0) \
    .filter(F.col('trip_time_cal') > 0) \
    .filter(F.col('base_passenger_fare') > 0) \
    .filter(F.col('trip_speed') > 1) \
    .filter(F.col('driver_pay') > 0) \
    .filter(F.col('pickup_datetime') >= F.col('request_datetime')) \
    .filter(F.col('trip_miles') <= 200) \
    .filter(F.col('trip_time_cal') <= 10800) \
    .filter(F.col('trip_speed') <= 120) \
    .withColumn('cbd_congestion_fee',
        F.when(F.col('cbd_congestion_fee').isNull(), 0)
         .otherwise(F.col('cbd_congestion_fee')))

print('Cleaning complete!')
# clean_count = df_clean.count()
# removed = raw_count - clean_count
# print(f'Rows after cleaning: {clean_count:,}')
# print(f'Rows removed:        {removed:,} ({removed/raw_count*100:.2f}%)')

## Derive features


In [ ]:
df_feat = df_clean \
    .withColumn('year',         F.year('pickup_datetime')) \
    .withColumn('month',        F.month('pickup_datetime')) \
    .withColumn('week',         F.weekofyear('pickup_datetime')) \
    .withColumn('date',         F.to_date('pickup_datetime')) \
    .withColumn('hour',         F.hour('pickup_datetime')) \
    .withColumn('do_hour', F.hour('dropoff_datetime'))\
    .withColumn('day_of_week',  F.dayofweek('pickup_datetime')) \
    .withColumn('is_weekend',   F.when(F.dayofweek('pickup_datetime').isin([1, 7]), 1).otherwise(0)) \
    .withColumn('15min_interval', (F.minute('pickup_datetime') / 15).cast('integer') * 15) \
    .withColumn('policy_period',
        F.when(F.col('pickup_datetime') < F.lit(POLICY_DATE), 'pre_policy')
         .otherwise('post_policy')) \
    .withColumn('wait_time_sec',
        F.unix_timestamp('pickup_datetime') - F.unix_timestamp('request_datetime')) \
    .withColumn('trip_duration_min', F.col('trip_time_cal') / 60) \
    .withColumn('total_fare',
        F.col('base_passenger_fare') + F.col('tolls') + F.col('tips') +
        F.col('congestion_surcharge') + F.col('sales_tax') + F.col('bcf') +
        F.col('airport_fee') + F.col('cbd_congestion_fee')) \
    .withColumn('tnc_comp',
        F.col('base_passenger_fare') + F.col('tips') - F.col('driver_pay'))

# Cache for repeated aggregations
df_feat.cache()
print('Feature engineering complete!')
new_cols = ['year', 'month', 'week', 'date', 'hour', 'day_of_week', 'is_weekend',
            '15min_interval', 'policy_period', 'wait_time_sec',
            'trip_duration_min', 'total_fare', 'tnc_comp']
print('New columns added:', new_cols)
# for c in new_cols:
#     print(f'  + {c}')

## Metric helpers


In [ ]:
def compute_metrics(df, group_cols):
    """
    Aggregate all metrics using aggregate-first approach.
    Group by group_cols and compute all metrics.
    """
    return df.groupBy(group_cols).agg(
        # Demand
        F.count('*').alias('trip_count'),

        # Intermediate sums for rate calculations
        F.sum('wait_time_sec').alias('_sum_wait_time'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('tnc_comp').alias('_sum_tnc_comp'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min')
    ).withColumn(
        'avg_wait_time_sec',     F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).withColumn(
        'base_fare_per_mile',    F.round(F.col('_sum_base_fare') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'base_fare_per_min',     F.round(F.col('_sum_base_fare') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'base_fare_per_trip',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'total_fare_per_trip',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'driver_pay_per_mile',   F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'driver_pay_per_min',    F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'driver_pay_per_trip',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'tnc_commission_rate',   F.round(F.col('_sum_tnc_comp') / F.col('_sum_base_fare'), 4)
    ).drop(
        '_sum_wait_time', '_sum_base_fare', '_sum_total_fare',
        '_sum_driver_pay', '_sum_tnc_comp', '_sum_trip_miles', '_sum_trip_duration_min'
    )


def compute_waittime_only(df, group_cols):
    """
    Aggregate wait time only — used for 15min granularity files.
    """
    return df.groupBy(group_cols).agg(
        F.count('*').alias('trip_count'),
        F.sum('wait_time_sec').alias('_sum_wait_time')
    ).withColumn(
        'avg_wait_time_sec', F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).drop('_sum_wait_time')


print('Aggregation helpers defined!')

## Pickup zone × hour


In [ ]:
# plan a: focusing on all platforms 

print('Aggregating by zone × date × hour...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )
    
        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period']
        df_zone_hour = compute_metrics(df_month, group_cols).orderBy('PULocationID', 'date', 'hour')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_hour_{year}_{month:02d}.parquet')
        df_zone_hour.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_zone_hour.unpersist()

## Dropoff zone × hour


In [ ]:
# plan a: all platforms DOLocationID x data x hour

print('Aggregating by dropoff zone × date × hour...')

for year in [2024, 2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = [
            'DOLocationID', 'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'do_hour', 'policy_period'
        ]
        df_DO_zone_hour = compute_metrics(df_month, group_cols) \
            .orderBy('DOLocationID', 'date', 'do_hour')

        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_DOzone_hour_{year}_{month:02d}.parquet')
        df_DO_zone_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)

        df_month.unpersist()
        df_DO_zone_hour.unpersist()

## OD pair × hour


In [ ]:
print('Aggregating by OD pair × date × hour...')

for year in [2024, 2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month:02d}...')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = [
            'PULocationID', 'DOLocationID',
            'date', 'year', 'month', 'week',
            'day_of_week', 'is_weekend', 'hour', 'policy_period'
        ]

        df_OD_hour = df_month.groupBy(group_cols).agg(
            F.count('*').alias('trip_count'),
            F.sum('base_passenger_fare').alias('_sum_base_fare'),
            F.sum('total_fare').alias('_sum_total_fare'),
            F.sum('trip_miles').alias('_sum_trip_miles'),
            F.sum('trip_duration_min').alias('_sum_trip_duration_min'),
            F.sum('cbd_congestion_fee').alias('total_cbd_fee')
        ).withColumn(
            'avg_base_fare',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_total_fare',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_trip_miles',   F.round(F.col('_sum_trip_miles') / F.col('trip_count'), 4)
        ).withColumn(
            'avg_duration_min', F.round(F.col('_sum_trip_duration_min') / F.col('trip_count'), 4)
        ).drop(
            '_sum_base_fare', '_sum_total_fare',
            '_sum_trip_miles', '_sum_trip_duration_min'
        ).orderBy('PULocationID', 'DOLocationID', 'date', 'hour')

        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_OD_hour_{year}_{month:02d}.parquet')
        df_OD_hour.write.mode('overwrite').parquet(output_file)
        print(f'Saved: {output_file}')

        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)

        df_month.unpersist()
        df_OD_hour.unpersist()

## Release the Spark session
Run after successful validation, before starting another preparation notebook.


In [ ]:
spark.stop()
